In [ ]:
%run ../shared_notebook_setup.py
import mlflow

# Specify the tracking server URI, e.g. http://localhost:5000
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("openai-api-basic-chat-observability")
print("MLflow experiment set: openai-api-basic-chat-observability")

In [ ]:
@mlflow.trace(name="chat_completion", span_type="LLM")
def _traced_chat_completion(messages, temperature):
    return client.chat.completions.create(
        model=DATABRICKS_ENDPOINT,
        messages=messages,
        temperature=temperature,
    )


def chat_once(user_message, history=None, temperature=0.2):
    """Send one chat request and log observability data to MLflow runs + traces."""
    history = history or []
    messages = history + [{"role": "user", "content": user_message}]

    with mlflow.start_run(run_name="openai_chat_turn") as run:
        response = _traced_chat_completion(messages=messages, temperature=temperature)

        assistant_text = response.choices[0].message.content or ""
        usage = response.usage

        mlflow.log_param("model", DATABRICKS_ENDPOINT)
        mlflow.log_param("temperature", temperature)
        mlflow.log_param("input_chars", len(user_message))
        mlflow.log_metric("prompt_tokens", usage.prompt_tokens if usage else 0)
        mlflow.log_metric("completion_tokens", usage.completion_tokens if usage else 0)
        mlflow.log_metric("total_tokens", usage.total_tokens if usage else 0)

        mlflow.log_text(user_message, "input_prompt.txt")
        mlflow.log_text(assistant_text, "assistant_response.txt")
        mlflow.set_tag("component", "chat")
        mlflow.set_tag("status", "success")

        print(f"MLflow run logged: {run.info.run_id}")
        print(f"MLflow trace logged: {mlflow.get_last_active_trace_id()}")

    updated_history = messages + [{"role": "assistant", "content": assistant_text}]
    return assistant_text, updated_history


In [ ]:
chat_history = []
assistant_reply, chat_history = chat_once(
    "Give me a two-line intro to MLflow observability.",
    history=chat_history,
    temperature=0.2,
 )
print("Assistant:", assistant_reply)

exp = mlflow.get_experiment_by_name("openai-api-basic-chat-observability")
if exp:
    traces = mlflow.search_traces(
        experiment_ids=[exp.experiment_id],
        max_results=5,
    )
    print("Recent traces:")
    display(traces)
else:
    print("Experiment not found.")

In [ ]:
assistant_reply, chat_history = chat_once("Now summarize that in one sentence.", history=chat_history)
print("Assistant:", assistant_reply)